# 01 — Cloud TPU: the big picture

**Learning goal.** Get a one-page mental model of what a Cloud TPU is, when to reach for one, and how the major versions compare on the dimensions that change a workload's behaviour.

**What you'll observe.**
- The catalog of Cloud TPU versions (v4, v5e, v5p, v6e) printed side by side.
- A full end-to-end simulation run — every stage of the lab in one cell — printed as a JSON summary.

**Knobs to tune.**
- `tpu_version` in the demo call: try `v4`, `v5e`, `v5p`, `v6e`. HBM, ICI bandwidth, and peak BF16 TFLOPS change.
- `chip_count`: how many chips participate. With 1 chip, all-reduce is free.
- `batch_size` / `hidden_size`: shape pressure on HBM and FLOPs.

**Failure modes to watch.**
- HBM utilisation above ~85% is the warning zone for OOM under load.
- v5e has the smallest HBM per chip — it's a great inference chip, but easy to OOM at training time with naive configs.
- Don't pick TPU just because it's fancy. For small models or low-throughput inference a CPU/GPU is often cheaper.

## Setup

We add the repo root to `sys.path` so `cloud_tpu_lab.src.*` imports resolve from inside the notebook directory.

In [ ]:
import sys, os
from pathlib import Path
# Add the parent of cloud_tpu_lab to sys.path so `cloud_tpu_lab.src.*` imports work.
HERE = Path(os.getcwd()).resolve()
_root = HERE
for _ in range(5):
    if (_root / "cloud_tpu_lab").exists():
        break
    _root = _root.parent
sys.path.insert(0, str(_root))
sys.path.insert(0, "..")
PLOT_DIR = Path("../artifacts/plots")
PLOT_DIR.mkdir(parents=True, exist_ok=True)
print("repo root:", _root)
print("plot dir:", PLOT_DIR.resolve())

## What is a Cloud TPU?

A **Cloud TPU** is a Google-designed ASIC for matrix-multiply-dominated workloads (deep learning training & inference). The unit you rent on Google Cloud is a **TPU VM**: a host machine attached to one or more TPU **chips**, connected to each other by a high-bandwidth Inter-Chip Interconnect (ICI). Larger configurations are called **slices** of a **pod**.

Pick a TPU when:

- The model is matmul-heavy (transformers, large MLPs, CNNs).
- You want batch-parallel throughput more than single-query latency.
- You can express the model in **XLA-compatible** frameworks: JAX, PyTorch/XLA, TensorFlow.

Skip a TPU when:

- The workload is host-bound (data loading dominates).
- You need exotic kernels XLA can't lower well.
- You only need one tiny inference call per minute.

This notebook focuses on **Google Cloud TPUs only** (not Edge / Coral / mobile TPUs).

## Cloud TPU version comparison

The catalog in `src/tpu_versions/cloud_tpu_catalog.py` tracks **public** numbers from cloud.google.com/tpu. The `render_table()` helper prints them side by side.

In [ ]:
from cloud_tpu_lab.src.tpu_versions.version_compare import render_table
from cloud_tpu_lab.src.tpu_versions.cloud_tpu_catalog import list_versions, get_spec

print("Known versions:", list_versions())
print()
print(render_table())

### Quick read of the table

- **v4** — older, 32 GB HBM, big 3D-torus pods. Workhorse for training at scale.
- **v5e** — cost-optimised, smallest HBM (16 GB), great $/throughput for inference and medium training.
- **v5p** — performance-optimised, 96 GB HBM, fastest BF16 of the v5 family.
- **v6e (Trillium)** — newest, ~4x peak vs v5e, 32 GB HBM, faster ICI.

## One full end-to-end simulation run

The function `examples.run_cpu_simulation_demo.run_demo` runs the whole vertical slice on CPU: build the tiny MLP, lower to fake HLO, simulate compile, place on fake devices, allocate HBM, shard, run N training steps, write logs/metrics/trace, and emit a cost report.

We keep the parameters tiny so this runs in well under 30 seconds.

In [ ]:
from cloud_tpu_lab.examples.run_cpu_simulation_demo import run_demo
from cloud_tpu_lab.src.common.config import WorkloadConfig, SimulationConfig

workload = WorkloadConfig(
    name="nb01_demo",
    tpu_version="v5e",
    chip_count=1,
    batch_size=8,
    hidden_size=128,
    num_layers=2,
    n_steps=3,
)
sim = SimulationConfig()
summary = run_demo(workload, sim, quiet=True)

import json as _json
print(_json.dumps(summary, indent=2))

## What you should see

- `trace_id` like `TRACE-0001`. Every artefact for this run is keyed on it.
- `n_log_events`, `n_metric_rows`, `n_trace_events` — the observability outputs.
- `compile_time_s` — first-step compile (cache miss). On rerun in the same process it falls to zero.
- `median_step_s` — steady-state step time, excluding the cold first step.
- `total_run_usd` — the cost estimate (placeholder hourly rate — update from cloud.google.com/tpu/pricing).
- `artifacts` — paths to the JSONL log, CSV metrics, Chrome-trace JSON, Markdown report.

You can open the report in any Markdown viewer and the trace JSON in `chrome://tracing`.

## Exercises

1. Re-run the cell above with `tpu_version="v5p"`. How does `total_run_usd` change? Is HBM utilisation higher or lower? Why?
2. Set `chip_count=4`. Compare `median_step_s` to the 1-chip version — does it scale linearly? Look at the breakdown in the printed report.
3. Bump `hidden_size` to 1024 and `num_layers` to 8. At what point does HBM utilisation cross 50% on v5e?
4. Try `precision="fp32"` in the workload. By how much does activation memory grow?
5. Open the produced Markdown report (`summary['artifacts']['report']`) and identify which stage dominated the run.